# Rift 01: a pickle file is basically a little program

Quick note to self before I start. This whole project rests on one weird fact: when you load a lot of AI model files, you are not just reading data, you are running code that shipped with the file. PyTorch models use a Python format called pickle, and pickle can carry instructions, not only values.

I want to actually watch that happen instead of taking it on faith. So this notebook does two things. First it looks at a normal, boring pickle. Then it makes one that runs code the moment I load it.

Nothing in here is harmful. The only thing my "sneaky" pickle does is call `print()`. That is the whole point: if a harmless pickle can make Python run `print()` just by being opened, an attacker can put something far worse than `print()` in that same slot.

In [ ]:
import pickle
import pickletools

## First, an honest pickle

Let me start with something completely ordinary, just a dictionary. I will turn it into pickle bytes (roughly what `torch.save` does when it writes a model), and then look at what those bytes actually are.

In [ ]:
data = {"user": "isuru", "role": "analyst", "clearance": 3}

blob_a = pickle.dumps(data)
blob_a

Those bytes look like noise, and that is exactly the problem. Nobody can glance at that and tell whether it is safe. So instead of guessing, I will disassemble it. `pickletools.dis` reads the instructions inside without running them, which is the same look-but-do-not-touch trick the scanner will rely on later.

In [ ]:
pickletools.dis(blob_a)

Read that top to bottom. Every instruction is just rebuilding the dictionary: push a key, push a value, put it in the dict. There is not a single function call anywhere. This is what a clean file looks like, and it is the baseline I want to compare against.

## Now a pickle that bites back

Here is the trick. Python lets a class define a method called `__reduce__` that says how to rebuild the object. Pickle trusts it completely: when the file is loaded, pickle calls whatever function `__reduce__` returns, with whatever arguments it returns.

So I am going to hand it `print`. When someone loads this pickle, Python runs print for me, even though loading is supposed to be "just reading data".

In [ ]:
class Sneaky:
    def __reduce__(self):
        # pickle will CALL the first item with the second item on load.
        # I am using print so this stays harmless. A real payload would
        # swap print for something like os.system. Same mechanism.
        return (print, ("  >>> code ran the moment you loaded me <<<",))

blob_b = pickle.dumps(Sneaky())

Let me disassemble this one and put it next to the honest pickle from before.

In [ ]:
pickletools.dis(blob_b)

There it is. Two instructions that were nowhere in the clean version:

- `GLOBAL` (or `STACK_GLOBAL`) pulls in a reference to the `print` function.
- `REDUCE` calls it.

That `REDUCE` is the entire vulnerability in one opcode. Any time a pickle reaches for a function and then calls it, that is the thing worth flagging. My scanner is really just going to hunt for this pattern, done well enough that people cannot hide it from me.

Last bit. I will load the pickle and run no code of my own. If the theory holds, a function should fire all by itself.

In [ ]:
result = pickle.loads(blob_b)

And there is the message, printed by the act of loading alone. I never called print myself. Loading the "data" executed code.

So the thing to remember: a pickle is not a document, it is a set of instructions, and opening one runs them. That is why downloading a random model file is closer to running a random `.exe` than most people realise, and it is the whole reason this project exists.

Next up: how PyTorch wraps these pickles inside a zip file, and why the scanner and the loader can end up disagreeing about what is inside. That disagreement is where the name Rift comes from.